# Module 08: QJL -- 1-Bit Quantized Johnson-Lindenstrauss

**Time**: ~35 minutes

**Prerequisites**: Module 07 (random projections and the JL lemma). Comfortable with PyTorch.

**Goal**: Take random projections to the extreme -- keep only the SIGN of each projection (1 bit instead of 32). Discover that this somehow still produces unbiased inner product estimates, build QJL from scratch, and connect to the reference implementation.

---

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import check_intuition, check_close, section_header

import math
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

torch.manual_seed(42)

## 1. The Extreme Question

In Module 07, we projected vectors using a random Gaussian matrix $R$ and stored the result in **full precision** -- 32-bit floats. That gave us the JL guarantee: distances are approximately preserved with $O(\log n / \varepsilon^2)$ dimensions.

But those projections still use 32 bits per coordinate. What if we kept only the **sign** of each projection?

- Full precision: $z = Rx$ stored as float32 --> 32 bits per projection
- Sign only: $\mathrm{sign}(z)$ --> **1 bit** per projection

That is a **32x compression** on top of the dimensionality reduction. We have thrown away ALL magnitude information -- only keeping whether each projection is positive or negative.

The question: does it still work? Can we recover meaningful inner product estimates from just the signs?

In [ ]:
section_header("The Extreme Question", "1 bit per projection -- can it work?")

# Let's see what sign quantization looks like
d = 128
m = 256  # projection dimension

gen = torch.Generator(); gen.manual_seed(42)
R = torch.randn(m, d, generator=gen)

# A random vector
x = torch.randn(d)

# Full precision projection
z_full = R @ x

# Sign-only projection
z_sign = torch.sign(z_full)  # each entry is +1 or -1

print(f"Full precision projection (first 10): {z_full[:10].tolist()}")
print(f"Sign-only projection (first 10):      {z_sign[:10].tolist()}")
print()
print(f"Full precision: {m * 32} bits = {m * 4} bytes")
print(f"Sign only:      {m * 1} bits  = {m // 8} bytes")
print(f"Compression:    {32}x")
print()
print("We have completely discarded all magnitude information.")
print("Each coordinate just says: 'positive' or 'negative'.")

## 2. Sign Quantization

Let's formalize what we are doing:

1. **Project**: $z = Rx$ where $R \in \mathbb{R}^{m \times d}$ has i.i.d. $\mathcal{N}(0,1)$ entries
2. **Keep only the sign**: $\mathrm{sign}(z_i) = \begin{cases} +1 & \text{if } z_i \geq 0 \\ -1 & \text{if } z_i < 0 \end{cases}$

We store one bit per projection: 0 for negative, 1 for non-negative. Plus we store the original norm $\|x\|$ (one float per vector).

This is incredibly aggressive. Each sign bit tells us only **which side of a random hyperplane** the vector $x$ falls on. The hyperplane is defined by the corresponding row of $R$.

Let's visualize this in 2D to build intuition.

In [ ]:
section_header("Sign Quantization in 2D", "Which side of each hyperplane?")

# 2D example: project onto random directions, keep only the sign
x_2d = torch.tensor([2.0, 1.0])
y_2d = torch.tensor([1.5, 1.8])

# 6 random projection directions
gen2d = torch.Generator(); gen2d.manual_seed(7)
R_2d = torch.randn(6, 2, generator=gen2d)
# Normalize rows for visualization (doesn't change the sign)
R_2d_norm = R_2d / R_2d.norm(dim=1, keepdim=True)

signs_x = (R_2d @ x_2d >= 0).int()  # {0, 1}
signs_y = (R_2d @ y_2d >= 0).int()

print(f"x = {x_2d.tolist()}, y = {y_2d.tolist()}")
print(f"True <x,y> = {torch.dot(x_2d, y_2d).item():.4f}")
print()
print("Hyperplane signs (0=negative, 1=non-negative):")
print(f"  x signs: {signs_x.tolist()}")
print(f"  y signs: {signs_y.tolist()}")
agree = (signs_x == signs_y).sum().item()
print(f"  Agreement: {agree}/{len(signs_x)} hyperplanes")
print()
print("Vectors on the SAME side of a hyperplane are more likely to be similar.")
print("The fraction of agreements relates to the angle between x and y.")

# Visualize
fig = go.Figure()

# Draw vectors
for v, name, color in [(x_2d, "x", "#636EFA"), (y_2d, "y", "#EF553B")]:
    fig.add_trace(go.Scatter(
        x=[0, v[0].item()], y=[0, v[1].item()],
        mode="lines+markers",
        name=name,
        line=dict(color=color, width=3),
        marker=dict(size=[0, 10]),
    ))

# Draw hyperplanes (perpendicular to each row of R)
colors_hp = ["#00CC96", "#AB63FA", "#FFA15A", "#19D3F3", "#FF6692", "#B6E880"]
for i, (r, c) in enumerate(zip(R_2d_norm, colors_hp)):
    # Hyperplane is perpendicular to r, passing through origin
    perp = torch.tensor([-r[1].item(), r[0].item()])
    t = 3.0
    sx = "+" if signs_x[i] else "-"
    sy = "+" if signs_y[i] else "-"
    fig.add_trace(go.Scatter(
        x=[-t * perp[0].item(), t * perp[0].item()],
        y=[-t * perp[1].item(), t * perp[1].item()],
        mode="lines",
        name=f"h{i} (x:{sx} y:{sy})",
        line=dict(color=c, width=1, dash="dash"),
    ))

fig.update_layout(
    title="Sign quantization: which side of each hyperplane?",
    xaxis=dict(scaleanchor="y", range=[-3, 3]),
    yaxis=dict(range=[-3, 3]),
    template="plotly_white",
    height=500,
    width=600,
)
fig.show()

## 3. The Surprising Result -- Unbiased Inner Product Estimation

Here is the remarkable fact. For a random Gaussian vector $r \sim \mathcal{N}(0, I)$:

$$\mathbb{E}[\mathrm{sign}(r \cdot x) \cdot (r \cdot y)] = \sqrt{\frac{2}{\pi}} \cdot \frac{\langle x, y \rangle}{\|x\|}$$

After scaling by $\|x\| \cdot \sqrt{\pi/2}$, we get:

$$\mathbb{E}\left[\|x\| \cdot \sqrt{\frac{\pi}{2}} \cdot \mathrm{sign}(r \cdot x) \cdot (r \cdot y)\right] = \langle x, y \rangle$$

This is an **unbiased estimator** of the inner product $\langle x, y \rangle$.

"Unbiased" means: on average, over many random projections, the estimate equals the true value. Individual estimates are noisy, but they are **centered** on the correct answer.

### Intuition

- $\mathrm{sign}(r \cdot x)$ tells you which side of a random hyperplane (normal to $r$) the vector $x$ falls on.
- $r \cdot y$ measures how far $y$ projects along $r$.
- The product $\mathrm{sign}(r \cdot x) \cdot (r \cdot y)$ flips the sign of $r \cdot y$ depending on where $x$ is.
- When $x$ and $y$ point in similar directions, the sign of $r \cdot x$ tends to agree with the sign of $r \cdot y$, making the product positive.
- Averaging over many random $r$ vectors extracts the inner product.

Let's verify this empirically.

In [ ]:
section_header("Verifying Unbiasedness", "The core mathematical result")

# Create two vectors with a known inner product
d = 64
torch.manual_seed(123)
x = torch.randn(d)
y = torch.randn(d)

true_ip = torch.dot(x, y).item()
x_norm = x.norm().item()

print(f"x norm: {x_norm:.4f}")
print(f"y norm: {y.norm().item():.4f}")
print(f"True <x, y> = {true_ip:.4f}")
print()

# For a single random direction r:
# E[sign(r.x) * (r.y)] = sqrt(2/pi) * <x,y> / ||x||
# So the scaled estimator is: ||x|| * sqrt(pi/2) * sign(r.x) * (r.y)
# Averaging over m directions: ||x|| * sqrt(pi/2) / m * sum_i sign(r_i.x) * (r_i.y)

m = 256
gen = torch.Generator(); gen.manual_seed(99)
R = torch.randn(m, d, generator=gen)

proj_x = R @ x          # shape (m,)
proj_y = R @ y          # shape (m,)
sign_x = torch.sign(proj_x)  # {-1, +1}

# Raw average of sign(r.x) * (r.y)
raw = (sign_x * proj_y).mean().item()
print(f"Raw average sign(r.x)*(r.y): {raw:.4f}")
print(f"Expected (sqrt(2/pi) * <x,y> / ||x||): {math.sqrt(2/math.pi) * true_ip / x_norm:.4f}")
print()

# Scaled estimator
estimate = x_norm * math.sqrt(math.pi / 2) * raw
print(f"Scaled estimate: {estimate:.4f}")
print(f"True <x,y>:      {true_ip:.4f}")
print(f"Error:            {abs(estimate - true_ip):.4f}")

One trial gives a noisy estimate. The power of unbiasedness is that over **many trials**, the estimates are centered on the true value. Let's see this with 1000 independent random matrices.

In [ ]:
section_header("1000 Trials", "Empirical verification of unbiasedness")

n_trials = 1000
m = 256
estimates = []

for trial in range(n_trials):
    gen_t = torch.Generator(); gen_t.manual_seed(trial)
    R_t = torch.randn(m, d, generator=gen_t)
    
    proj_x_t = R_t @ x
    proj_y_t = R_t @ y
    sign_x_t = torch.sign(proj_x_t)
    
    raw_t = (sign_x_t * proj_y_t).mean().item()
    est_t = x.norm().item() * math.sqrt(math.pi / 2) * raw_t
    estimates.append(est_t)

estimates = np.array(estimates)

print(f"True <x, y>:       {true_ip:.4f}")
print(f"Mean estimate:     {estimates.mean():.4f}")
print(f"Std of estimates:  {estimates.std():.4f}")
print(f"Mean - True:       {estimates.mean() - true_ip:.4f}  (bias -- should be near 0)")
print()
print("The mean is very close to the true value. That is unbiasedness.")
print("Individual estimates scatter around it, but the CENTER is correct.")

In [ ]:
# Histogram of estimates
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=estimates,
    nbinsx=50,
    name="QJL estimates",
    marker_color="#636EFA",
    opacity=0.8,
))

fig.add_vline(
    x=true_ip,
    line_color="red",
    line_width=3,
    annotation_text=f"True <x,y> = {true_ip:.2f}",
    annotation_position="top right",
)

fig.add_vline(
    x=estimates.mean(),
    line_color="green",
    line_width=2,
    line_dash="dash",
    annotation_text=f"Mean estimate = {estimates.mean():.2f}",
    annotation_position="top left",
)

fig.update_layout(
    title="QJL Estimates over 1000 Trials (m=256 projections each)",
    xaxis_title="Estimated <x, y>",
    yaxis_title="Count",
    template="plotly_white",
    height=450,
    width=700,
)
fig.show()

print(f"The histogram is centered on the true value (red line).")
print(f"The green dashed line shows the mean of 1000 estimates.")

## 4. The sqrt(pi/2) Factor -- Where Does It Come From?

The correction factor $\sqrt{\pi/2} \approx 1.2533$ has an elegant origin.

For a standard normal random variable $r \sim \mathcal{N}(0, 1)$:

$$\mathbb{E}[|r|] = \sqrt{\frac{2}{\pi}}$$

This is the mean of the **folded normal distribution** -- what you get when you take the absolute value of a Gaussian.

When we take $\mathrm{sign}(r \cdot x)$, we discard the magnitude $|r \cdot x|$ and keep only the sign. The expected magnitude we threw away is $\sqrt{2/\pi} \cdot \|x\|$.

To compensate, we multiply by $\|x\| \cdot \sqrt{\pi/2}$, which is exactly the reciprocal of $\sqrt{2/\pi} / \|x\|$.

Let's verify this empirically.

In [ ]:
section_header("The sqrt(pi/2) Factor", "From the folded normal distribution")

# E[|r|] for r ~ N(0,1) should be sqrt(2/pi)
gen_fold = torch.Generator(); gen_fold.manual_seed(0)
r_samples = torch.randn(100000, generator=gen_fold)

empirical_mean_abs = r_samples.abs().mean().item()
theoretical = math.sqrt(2 / math.pi)

print(f"E[|r|] empirical (100k samples): {empirical_mean_abs:.6f}")
print(f"E[|r|] theoretical (sqrt(2/pi)): {theoretical:.6f}")
print()
print(f"Correction factor sqrt(pi/2):    {math.sqrt(math.pi / 2):.6f}")
print(f"Check: sqrt(2/pi) * sqrt(pi/2) = {theoretical * math.sqrt(math.pi / 2):.6f}  (should be 1.0)")
print()
print("When we take sign(r.x), we lose the factor E[|r.x|] = sqrt(2/pi) * ||x||.")
print("Multiplying by ||x|| * sqrt(pi/2) exactly compensates for this loss.")

## 5. Build QJL from Scratch

Now let's put it all together into a working QJL class. The key ideas:

1. **Quantize**: project with $R$, store only the sign bits + original norms
2. **Estimate inner product**: project the query in full precision, dot with sign bits, scale

This is **asymmetric** estimation -- the database vectors are quantized to 1 bit, but the query stays at full precision.

In [ ]:
section_header("Building QJL from Scratch", "The complete 1-bit quantizer")

class SimpleQJL:
    """Minimal QJL implementation for learning.
    
    Projects vectors with a random Gaussian matrix and stores only the
    sign bits (1 bit per projection) plus the original L2 norm.
    """
    
    def __init__(self, input_dim, proj_dim, seed=42):
        gen = torch.Generator(); gen.manual_seed(seed)
        self.R = torch.randn(proj_dim, input_dim, generator=gen)
        self.proj_dim = proj_dim
    
    def quantize(self, x):
        """Quantize x to sign bits + norms.
        
        Args:
            x: tensor of shape (..., input_dim)
        
        Returns:
            sign_bits: uint8 tensor of 0s and 1s, shape (..., proj_dim)
            norms: L2 norms, shape (...)
        """
        norms = x.norm(dim=-1)
        z = x @ self.R.t()  # full precision projection
        sign_bits = (z >= 0).to(torch.uint8)  # 1 bit per projection
        return sign_bits, norms
    
    def estimate_ip(self, query, sign_bits, norms):
        """Estimate inner products between query and quantized vectors.
        
        ASYMMETRIC: query is projected in full precision, database is 1-bit.
        
        Args:
            query: tensor of shape (input_dim,) or (nq, input_dim)
            sign_bits: uint8 tensor, shape (n, proj_dim)
            norms: L2 norms, shape (n,)
        
        Returns:
            Estimated inner products, shape (..., n)
        """
        query_proj = query @ self.R.t()  # full precision query projection
        signs = sign_bits.float() * 2 - 1  # {0,1} -> {-1,+1}
        raw = query_proj @ signs.t() / self.proj_dim
        return raw * norms * math.sqrt(math.pi / 2)

print("SimpleQJL class defined.")
print("  - quantize(): stores sign bits (1-bit) + norms")
print("  - estimate_ip(): asymmetric inner product estimation")

In [ ]:
# Test it on a small example
d = 128
m = 256
n_db = 50  # database vectors

torch.manual_seed(42)
database = torch.randn(n_db, d)
query = torch.randn(d)

# True inner products
true_ips = query @ database.t()  # shape (n_db,)

# QJL estimates
qjl = SimpleQJL(input_dim=d, proj_dim=m, seed=0)
sign_bits, norms = qjl.quantize(database)
est_ips = qjl.estimate_ip(query, sign_bits, norms).squeeze()

print(f"Database: {n_db} vectors of dim {d}")
print(f"Projection dim: {m}")
print(f"Sign bits shape: {sign_bits.shape} (dtype: {sign_bits.dtype})")
print(f"Norms shape: {norms.shape}")
print()
print("Memory per database vector:")
print(f"  Full precision: {d * 4} bytes")
print(f"  QJL (sign bits + norm): {m // 8} + 4 = {m // 8 + 4} bytes")
print(f"  Compression ratio: {d * 4 / (m // 8 + 4):.1f}x")
print()

# How good are the estimates?
correlation = torch.corrcoef(torch.stack([true_ips, est_ips]))[0, 1].item()
mae = (true_ips - est_ips).abs().mean().item()
print(f"Correlation between true and estimated IPs: {correlation:.4f}")
print(f"Mean absolute error: {mae:.4f}")

In [ ]:
# Scatter plot: true vs estimated inner products
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=true_ips.numpy(),
    y=est_ips.detach().numpy(),
    mode="markers",
    marker=dict(size=8, color="#636EFA", opacity=0.7),
    name="QJL estimates",
))

# Perfect line
ip_range = [true_ips.min().item() - 2, true_ips.max().item() + 2]
fig.add_trace(go.Scatter(
    x=ip_range, y=ip_range,
    mode="lines",
    line=dict(color="red", dash="dash", width=2),
    name="Perfect estimation",
))

fig.update_layout(
    title=f"True vs QJL-Estimated Inner Products (m={m}, corr={correlation:.3f})",
    xaxis_title="True <query, db_vector>",
    yaxis_title="QJL estimate",
    template="plotly_white",
    height=500,
    width=600,
)
fig.show()

## 6. Verify Unbiasedness Empirically

Let's do a rigorous test. Create two specific vectors with a known inner product, run QJL estimation 1000 times with different random seeds, and verify the estimates are centered on the true value.

In [ ]:
section_header("Rigorous Unbiasedness Test", "1000 independent QJL instances")

d = 128
m = 128  # proj_dim = input_dim

# Two vectors with known properties
torch.manual_seed(55)
x = torch.randn(d)
y = torch.randn(d)
true_ip = torch.dot(x, y).item()

print(f"||x|| = {x.norm().item():.4f}")
print(f"||y|| = {y.norm().item():.4f}")
print(f"True <x, y> = {true_ip:.4f}")
print()

# Run 1000 independent QJL estimations
n_trials = 1000
estimates_unbias = []

for seed in range(n_trials):
    qjl_t = SimpleQJL(input_dim=d, proj_dim=m, seed=seed)
    sb, nm = qjl_t.quantize(x.unsqueeze(0))  # quantize x
    est = qjl_t.estimate_ip(y.unsqueeze(0), sb, nm).item()  # estimate <y, x>
    estimates_unbias.append(est)

estimates_unbias = np.array(estimates_unbias)

print(f"Mean estimate:  {estimates_unbias.mean():.4f}")
print(f"True value:     {true_ip:.4f}")
print(f"Bias:           {estimates_unbias.mean() - true_ip:.4f}")
print(f"Std:            {estimates_unbias.std():.4f}")
print()

# Check unbiasedness
check_close(
    estimates_unbias.mean(), true_ip,
    name="Mean QJL estimate vs true IP",
    rtol=0.05, atol=0.5,
)

In [ ]:
# Histogram with true value marked
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=estimates_unbias,
    nbinsx=60,
    name="QJL estimates (1000 trials)",
    marker_color="#636EFA",
    opacity=0.8,
))

fig.add_vline(
    x=true_ip,
    line_color="red",
    line_width=3,
    annotation_text=f"True <x,y> = {true_ip:.2f}",
    annotation_position="top right",
)

fig.add_vline(
    x=estimates_unbias.mean(),
    line_color="green",
    line_width=2,
    line_dash="dash",
    annotation_text=f"Mean = {estimates_unbias.mean():.2f}",
    annotation_position="top left",
)

fig.update_layout(
    title=f"Unbiasedness: 1000 QJL Estimates (m={m}, d={d})",
    xaxis_title="Estimated inner product",
    yaxis_title="Count",
    template="plotly_white",
    height=450,
    width=700,
)
fig.show()

print("Key insight: individual estimates are noisy, but they are CENTERED correctly.")
print("This is what 'unbiased' means -- the expected value equals the true value.")

## 7. Asymmetric Estimation

A critical design choice in QJL is that the estimation is **asymmetric**:

- **Database vectors** (keys in a KV cache): quantized to 1-bit sign codes
- **Query vector**: projected in **full precision** (no quantization)

Why not quantize both sides? Let's compare.

### Symmetric (quantize both) vs Asymmetric (quantize only database)

If we quantize both $x$ and $y$ to signs, we get:
$$\hat{\langle x, y \rangle}_{\text{sym}} = \|x\| \cdot \|y\| \cdot \frac{\pi}{2m} \sum_i \mathrm{sign}(r_i \cdot x) \cdot \mathrm{sign}(r_i \cdot y)$$

This is related to the **angular similarity** -- the fraction of hyperplanes that $x$ and $y$ agree on. It works, but it is noisier than the asymmetric version.

The asymmetric version keeps the full-precision query projection, which preserves more signal.

In [ ]:
section_header("Asymmetric vs Symmetric", "Why we only quantize the database")

d = 128
m = 128
n_db = 100

torch.manual_seed(42)
database = torch.randn(n_db, d)
query = torch.randn(d)

true_ips = query @ database.t()

# Asymmetric: quantize database, full-precision query
qjl_asym = SimpleQJL(input_dim=d, proj_dim=m, seed=0)
sb_db, nm_db = qjl_asym.quantize(database)
est_asym = qjl_asym.estimate_ip(query, sb_db, nm_db).squeeze()

# Symmetric: quantize both query and database
def symmetric_estimate(qjl, query, sign_bits_db, norms_db):
    """Both sides quantized to signs."""
    query_proj = query @ qjl.R.t()
    query_signs = (query_proj >= 0).float() * 2 - 1  # quantize query too!
    db_signs = sign_bits_db.float() * 2 - 1
    # Agreement fraction * scaling
    raw = query_signs @ db_signs.t() / qjl.proj_dim
    # For symmetric: scale by ||x|| * ||y|| * pi/2
    query_norm = query.norm()
    return raw * norms_db * query_norm * (math.pi / 2)

est_sym = symmetric_estimate(qjl_asym, query, sb_db, nm_db).squeeze()

# Compare
mae_asym = (true_ips - est_asym).abs().mean().item()
mae_sym = (true_ips - est_sym).abs().mean().item()
corr_asym = torch.corrcoef(torch.stack([true_ips, est_asym]))[0, 1].item()
corr_sym = torch.corrcoef(torch.stack([true_ips, est_sym]))[0, 1].item()

print(f"{'Method':<15} {'MAE':>8} {'Correlation':>12}")
print("-" * 37)
print(f"{'Asymmetric':<15} {mae_asym:>8.4f} {corr_asym:>12.4f}")
print(f"{'Symmetric':<15} {mae_sym:>8.4f} {corr_sym:>12.4f}")
print()
print("Asymmetric is better: keeping the query at full precision preserves more signal.")
print("In practice: compress your KV cache (database), queries remain full precision.")

In [ ]:
# Side by side scatter plots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"Asymmetric (corr={corr_asym:.3f})", f"Symmetric (corr={corr_sym:.3f})"],
)

for col, (est, name) in enumerate([(est_asym, "Asymmetric"), (est_sym, "Symmetric")], 1):
    fig.add_trace(go.Scatter(
        x=true_ips.numpy(),
        y=est.detach().numpy(),
        mode="markers",
        marker=dict(size=5, opacity=0.6),
        name=name,
        showlegend=True,
    ), row=1, col=col)
    
    ip_range = [true_ips.min().item() - 2, true_ips.max().item() + 2]
    fig.add_trace(go.Scatter(
        x=ip_range, y=ip_range,
        mode="lines",
        line=dict(color="red", dash="dash", width=1),
        showlegend=False,
    ), row=1, col=col)

fig.update_layout(
    title="Asymmetric vs Symmetric QJL Estimation",
    template="plotly_white",
    height=450,
    width=900,
)
fig.update_xaxes(title_text="True inner product")
fig.update_yaxes(title_text="Estimated inner product", col=1)
fig.show()

## 8. Connect to Reference Code

Now let's look at the production QJL implementation in `reference-code/turboquant-torch/turboquant/qjl.py`. It has the same core logic as our `SimpleQJL`, with several engineering improvements.

```python
class QJL:
    def __init__(self, input_dim, proj_dim=None, seed=42, proj_type="gaussian"):
        self.input_dim = input_dim
        self.proj_dim = proj_dim or input_dim
        self.seed = seed
        self.proj_type = proj_type
        self._proj_matrix = None

    def quantize(self, x):
        R = self._get_proj_matrix(x.device, x.dtype)
        norms = torch.norm(x, dim=-1)
        z = x @ R.t()
        sign_bits = (z >= 0).to(torch.uint8)
        return QJLOutput(sign_bits=sign_bits, norms=norms)

    def dequantize_for_dot(self, output):
        R = self._get_proj_matrix(output.sign_bits.device, torch.float32)
        signs = output.sign_bits.float() * 2 - 1
        scale = output.norms.unsqueeze(-1) * math.sqrt(math.pi / 2) / self.proj_dim
        return scale * (signs @ R)

    def estimate_inner_product(self, query, output):
        R = self._get_proj_matrix(query.device, query.dtype)
        query_proj = query @ R.t()
        signs = output.sign_bits.float() * 2 - 1
        raw = query_proj @ signs.t() / self.proj_dim
        return raw * output.norms.unsqueeze(0) * math.sqrt(math.pi / 2)
```

### Key differences from our SimpleQJL:

**1. Lazy projection matrix generation** (`_get_proj_matrix`)

The projection matrix $R$ is not created in `__init__`. Instead, it is generated on first use and cached. This means:
- The constructor is cheap (no memory allocation)
- The matrix is generated on the correct device (CPU or GPU)
- The same seed always produces the same matrix, regardless of when it is created

**2. Device and dtype handling**

The `_get_proj_matrix` method accepts `device` and `dtype` parameters. If the cached matrix is on the wrong device (e.g., you moved data to GPU), it regenerates on the correct device. This is essential for GPU support.

**3. NamedTuple output** (`QJLOutput`)

Instead of returning a bare tuple `(sign_bits, norms)`, the reference code uses a `NamedTuple`. This makes the code self-documenting:
```python
output = qjl.quantize(x)
output.sign_bits  # clear what this is
output.norms      # clear what this is
```

**4. `dequantize_for_dot` method**

This reconstructs a full-dimensional vector from sign bits that can be used in standard dot products. It computes $\hat{x} = \|x\| \cdot \sqrt{\pi/2} / m \cdot \mathrm{signs} \cdot R$, which maps the 1-bit codes back to $d$-dimensional space. The dot product $\langle \hat{x}, y \rangle$ then estimates $\langle x, y \rangle$.

**5. Rademacher option**

The `proj_type` parameter allows using a Rademacher matrix (uniform $\{-1, +1\}$) instead of Gaussian. Rademacher matrices are faster to generate and have similar theoretical properties.

In [ ]:
section_header("Reference Code Comparison", "Verify our SimpleQJL matches the production code")

# Import the reference implementation
sys.path.insert(0, "../reference-code/turboquant-torch")
from turboquant.qjl import QJL, QJLOutput

d = 128
m = 256
seed = 42

torch.manual_seed(0)
x_test = torch.randn(10, d)
q_test = torch.randn(d)

# Reference implementation
ref_qjl = QJL(input_dim=d, proj_dim=m, seed=seed)
ref_output = ref_qjl.quantize(x_test)
ref_ips = ref_qjl.estimate_inner_product(q_test.unsqueeze(0), ref_output).squeeze(0)

# Our SimpleQJL with the same seed
simple = SimpleQJL(input_dim=d, proj_dim=m, seed=seed)
sb, nm = simple.quantize(x_test)
simple_ips = simple.estimate_ip(q_test.unsqueeze(0), sb, nm).squeeze(0)

# True inner products
true_test_ips = q_test @ x_test.t()

print("Sign bits match:", (ref_output.sign_bits == sb).all().item())
print("Norms match:", torch.allclose(ref_output.norms, nm))
print("IP estimates match:", torch.allclose(ref_ips, simple_ips, rtol=1e-4))
print()
print("QJLOutput type:", type(ref_output))
print("Fields:", ref_output._fields)
print()

# Also test dequantize_for_dot
reconstructed = ref_qjl.dequantize_for_dot(ref_output)
recon_ips = q_test @ reconstructed.t()

check_close(
    ref_ips, recon_ips,
    name="estimate_inner_product vs dequantize_for_dot",
    rtol=1e-4, atol=1e-4,
)
print("Both methods produce the same inner product estimates (as expected).")

## 9. Exercise: Variance of the QJL Estimator

The QJL estimator is unbiased -- its expected value equals the true inner product. But how **noisy** is it? The variance tells us how much individual estimates scatter around the true value.

**Setup**: Two vectors $x, y$ with:
- $\|x\| = 2.0$
- $\|y\| = 3.0$  
- $\langle x, y \rangle = 4.0$
- `proj_dim = input_dim = 128`

**Your task**:
1. Run 500 independent QJL trials (different seeds)
2. Compute the empirical variance of the estimates
3. Compare to the theoretical variance: $\text{Var} \approx \frac{\pi}{2m} \cdot \|x\|^2 \cdot \|y\|^2 \approx 0.110$

The theoretical formula comes from the variance of the sign-based estimator. The factor $\pi/(2m)$ captures the noise introduced by discarding magnitudes, and the dependence on $\|x\|^2 \cdot \|y\|^2$ means larger vectors produce noisier estimates.

In [ ]:
section_header("Exercise: Estimator Variance", "How noisy is the 1-bit estimator?")

d_ex = 128
m_ex = 128  # proj_dim = input_dim

# Construct x, y with specified norms and inner product
# x = [2, 0, 0, ...], y = [2, sqrt(5), 0, ...] gives ||x||=2, ||y||=3, <x,y>=4
x_ex = torch.zeros(d_ex)
x_ex[0] = 2.0

y_ex = torch.zeros(d_ex)
y_ex[0] = 2.0                         # contributes 2*2=4 to inner product
y_ex[1] = math.sqrt(9.0 - 4.0)        # ||y||^2 = 4 + 5 = 9, so ||y|| = 3

print(f"||x|| = {x_ex.norm().item():.4f}")
print(f"||y|| = {y_ex.norm().item():.4f}")
print(f"<x, y> = {torch.dot(x_ex, y_ex).item():.4f}")
print()

# ---- YOUR CODE HERE ----
# Run 500 QJL trials with different seeds.
# For each trial:
#   1. Create a SimpleQJL with seed=trial_number
#   2. Quantize x_ex
#   3. Estimate <y_ex, x_ex>
#   4. Store the estimate

n_trials_ex = 500
estimates_ex = []

for trial in range(n_trials_ex):
    qjl_ex = SimpleQJL(input_dim=d_ex, proj_dim=m_ex, seed=trial)
    sb_ex, nm_ex = qjl_ex.quantize(x_ex.unsqueeze(0))
    est_ex = qjl_ex.estimate_ip(y_ex.unsqueeze(0), sb_ex, nm_ex).item()
    estimates_ex.append(est_ex)

estimates_ex = np.array(estimates_ex)
# ---- END YOUR CODE ----

empirical_var = estimates_ex.var()
theoretical_var = (math.pi / (2 * m_ex)) * x_ex.norm().item()**2 * y_ex.norm().item()**2

print(f"Empirical variance:   {empirical_var:.4f}")
print(f"Theoretical variance: {theoretical_var:.4f}")
print(f"Ratio (empirical / theoretical): {empirical_var / theoretical_var:.3f}")
print()

check_intuition(
    result=empirical_var,
    expected=theoretical_var,
    hint="The theoretical variance is pi/(2m) * ||x||^2 * ||y||^2. Try more trials if the ratio is off.",
    tolerance=0.4,
    label="Empirical variance",
)

In [ ]:
# Visualize the exercise results
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=estimates_ex,
    nbinsx=40,
    name="QJL estimates (500 trials)",
    marker_color="#636EFA",
    opacity=0.8,
))

true_ip_ex = 4.0
fig.add_vline(
    x=true_ip_ex,
    line_color="red",
    line_width=3,
    annotation_text=f"True <x,y> = {true_ip_ex:.1f}",
    annotation_position="top right",
)

# Show +/- 1 std band
std_ex = np.sqrt(empirical_var)
fig.add_vrect(
    x0=true_ip_ex - std_ex,
    x1=true_ip_ex + std_ex,
    fillcolor="rgba(255, 0, 0, 0.1)",
    line_width=0,
    annotation_text=f"+/- 1 std = {std_ex:.3f}",
    annotation_position="top left",
)

fig.update_layout(
    title=f"Exercise: QJL Variance (m={m_ex}, empirical={empirical_var:.4f}, theoretical={theoretical_var:.4f})",
    xaxis_title="Estimated <x, y>",
    yaxis_title="Count",
    template="plotly_white",
    height=450,
    width=700,
)
fig.show()

print(f"The variance is roughly pi/(2*{m_ex}) * {x_ex.norm().item():.0f}^2 * {y_ex.norm().item():.0f}^2")
print(f"= {math.pi:.4f} / {2*m_ex} * {x_ex.norm().item()**2:.0f} * {y_ex.norm().item()**2:.0f}")
print(f"= {theoretical_var:.4f}")
print()
print("The noise decreases as 1/m. More projections = less variance = better estimates.")

## Summary

- **Sign quantization** keeps only 1 bit per projection -- the most extreme compression possible.
- Despite discarding all magnitude information, the **QJL estimator is unbiased**: on average, it recovers the true inner product exactly.
- The correction factor $\sqrt{\pi/2}$ comes from the expected magnitude of a folded normal distribution.
- **Asymmetric** estimation (quantize database, keep query at full precision) is significantly better than symmetric.
- The variance scales as $\frac{\pi}{2m} \cdot \|x\|^2 \cdot \|y\|^2$ -- more projections reduce noise.
- The reference `QJL` class adds lazy matrix generation, device handling, NamedTuple outputs, and a `dequantize_for_dot` method.

**The big picture**: QJL gives us a 1-bit representation of vectors that preserves inner product information. Combined with an MSE quantizer on the original vector, QJL on the residual produces TurboQuant's unbiased inner product estimator.

**Next**: Module 09 -- putting the pieces together into the full TurboQuant pipeline.